In [ ]:
# =========================
# nb_ingest_brown_shipley_v2
# Stage 1: Faithful Source Ingestion
# =========================
"""
Purpose: Provide a faithful representation of Brown Shipley source files
         in Fabric for Stage 2 processing.

File Structure:
- Row 1: Groupings (ignored)
- Row 2: Column headers
- Row 3+: Data
"""

# ---------- Establish Workspace Context ----------
# CRITICAL: Required when notebook is in a subfolder
# This ensures Fabric's lakehouse context is properly initialized
try:
    spark.sql("SELECT 1")
    print("[INFO] Workspace context initialized")
except Exception as e:
    print(f"[WARNING] Workspace context issue: {str(e)}")
    print("[ACTION] Ensure lakehouse is attached in Fabric notebook UI")

# ---------- Parameters ----------
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by nb_run_all in real run

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

dfm_id = "brown_shipley"
dfm_name = "Brown Shipley"

print(f"[INFO] Starting Brown Shipley ingestion for period={period}, run_id={run_id}")

In [ ]:
# ---------- Imports ----------
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
from datetime import datetime

# ---------- Ensure Workspace Context ----------
# This is required for notebooks in subfolders
try:
    # Re-establish lakehouse context
    lakehouse_id = spark.conf.get("spark.fabric.workspace.artifact.id")
    print(f"[INFO] Lakehouse context established: {lakehouse_id}")
except:
    print("[WARNING] Could not verify lakehouse context - ensure lakehouse is attached in Fabric UI")

# ---------- Paths ----------
landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
print(f"[INFO] Landing path: {landing_path}")

In [ ]:
# ---------- Helper Functions ----------

def normalize_column_name(col_name):
    """Normalize column names: remove special chars, lowercase, replace spaces with underscores."""
    import re
    normalized = str(col_name).strip().lower()
    normalized = re.sub(r'[^\w\s]', '', normalized)
    normalized = re.sub(r'\s+', '_', normalized)
    return normalized

def read_csv_with_header_row2(file_path, source_file_name):
    """
    Read CSV file where:
    - Row 1 = Groupings (skip)
    - Row 2 = Headers
    - Row 3+ = Data
    
    Returns: Spark DataFrame with all columns as strings
    """
    try:
        # Read with pandas, skip first row (groupings), use row 2 as header
        pdf = pd.read_csv(
            file_path,
            skiprows=1,  # Skip row 1 (groupings)
            dtype=str,   # Keep everything as string for fidelity
            encoding='utf-8',
            keep_default_na=False,  # Preserve empty strings
            na_filter=False
        )
        
        # Normalize column names
        pdf.columns = [normalize_column_name(col) for col in pdf.columns]
        
        # Remove any completely empty rows
        pdf = pdf[pdf.apply(lambda x: x.str.strip().str.len().sum(), axis=1) > 0]
        
        if len(pdf) == 0:
            print(f"[WARNING] No data rows found in {source_file_name}")
            return None
        
        # Convert to Spark DataFrame
        sdf = spark.createDataFrame(pdf)
        
        # Add source tracking
        sdf = sdf.withColumn("source_file", F.lit(source_file_name))
        
        print(f"[INFO] Successfully read {source_file_name}: {sdf.count()} rows, {len(sdf.columns)-1} columns")
        return sdf
        
    except Exception as e:
        print(f"[ERROR] Failed to read {source_file_name}: {str(e)}")
        return None

def add_row_identifiers(df, row_type):
    """Add unique row identifiers to each record."""
    w = Window.orderBy(F.monotonically_increasing_id())
    return (
        df.withColumn("_row_num", F.row_number().over(w))
          .withColumn(
              "source_row_id",
              F.concat_ws(":", 
                  F.col("source_file"),
                  F.lit(row_type),
                  F.col("_row_num").cast("string")
              )
          )
          .drop("_row_num")
    )

print("[INFO] Helper functions loaded")

In [ ]:
# ---------- Discover Files ----------
print("[STEP 1] Discovering source files...")

try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
    csv_files = [f for f in entries if f.name.lower().endswith(".csv")]
    print(f"[INFO] Found {len(csv_files)} CSV file(s)")
    for f in csv_files:
        print(f"  - {f.name}")
except Exception as e:
    print(f"[ERROR] Could not access landing path: {str(e)}")
    mssparkutils.notebook.exit("NO_FILES")

if len(csv_files) == 0:
    print("[ERROR] No CSV files found")
    mssparkutils.notebook.exit("NO_FILES")

In [ ]:
# ---------- Identify Expected Files ----------
print("[STEP 2] Identifying expected files...")

positions_file = next((f for f in csv_files if f.name.lower() == "notification.csv"), None)
cash_file = next((f for f in csv_files if f.name.lower() == "notification - cash.csv"), None)

if positions_file:
    print(f"[INFO] Positions file: {positions_file.name}")
else:
    print("[WARNING] Positions file (Notification.csv) not found")

if cash_file:
    print(f"[INFO] Cash file: {cash_file.name}")
else:
    print("[WARNING] Cash file (Notification - Cash.csv) not found")

if not positions_file and not cash_file:
    print("[ERROR] Neither expected file found")
    mssparkutils.notebook.exit("NO_FILES")

In [ ]:
# ---------- Read Positions File ----------
print("[STEP 3] Reading positions file...")

df_positions = None
if positions_file:
    df_positions = read_csv_with_header_row2(positions_file.path, positions_file.name)
    
    if df_positions:
        df_positions = add_row_identifiers(df_positions, "POS")
        df_positions = df_positions.withColumn("record_type", F.lit("POSITION"))
        print(f"[INFO] Positions file processed: {df_positions.count()} rows")
        print(f"[INFO] Columns: {', '.join(sorted([c for c in df_positions.columns if c not in ['source_file', 'source_row_id', 'record_type']]))}")
    else:
        print("[WARNING] Failed to process positions file")
else:
    print("[INFO] Positions file not found, skipping")

In [ ]:
# ---------- Read Cash File ----------
print("[STEP 4] Reading cash file...")

df_cash = None
if cash_file:
    df_cash = read_csv_with_header_row2(cash_file.path, cash_file.name)
    
    if df_cash:
        df_cash = add_row_identifiers(df_cash, "CASH")
        df_cash = df_cash.withColumn("record_type", F.lit("CASH"))
        print(f"[INFO] Cash file processed: {df_cash.count()} rows")
        print(f"[INFO] Columns: {', '.join(sorted([c for c in df_cash.columns if c not in ['source_file', 'source_row_id', 'record_type']]))}")
    else:
        print("[WARNING] Failed to process cash file")
else:
    print("[INFO] Cash file not found, skipping")

In [ ]:
# ---------- Profile the Data ----------
print("\n" + "=" * 80)
print("[DATA PROFILE] Source File Statistics")
print("=" * 80)

if df_positions:
    print(f"\nPOSITIONS FILE: {positions_file.name}")
    print(f"  Total rows: {df_positions.count()}")
    print(f"  Total columns: {len([c for c in df_positions.columns if c not in ['source_file', 'source_row_id', 'record_type']])}")
    
    # Check for key columns
    if 'client_id' in df_positions.columns:
        distinct_clients = df_positions.select("client_id").distinct().count()
        null_clients = df_positions.filter(F.col("client_id").isNull() | (F.trim(F.col("client_id")) == "")).count()
        print(f"  Distinct client_id values: {distinct_clients}")
        print(f"  Rows with empty client_id: {null_clients}")
    
    if 'isin_code' in df_positions.columns:
        distinct_isins = df_positions.select("isin_code").distinct().count()
        print(f"  Distinct ISIN codes: {distinct_isins}")

if df_cash:
    print(f"\nCASH FILE: {cash_file.name}")
    print(f"  Total rows: {df_cash.count()}")
    print(f"  Total columns: {len([c for c in df_cash.columns if c not in ['source_file', 'source_row_id', 'record_type']])}")
    
    # Check for key columns
    if 'client_id' in df_cash.columns:
        distinct_clients = df_cash.select("client_id").distinct().count()
        null_clients = df_cash.filter(F.col("client_id").isNull() | (F.trim(F.col("client_id")) == "")).count()
        print(f"  Distinct client_id values: {distinct_clients}")
        print(f"  Rows with empty client_id: {null_clients}")

print("=" * 80 + "\n")

In [ ]:
# ---------- Add Metadata Columns ----------
print("[STEP 5] Adding metadata columns...")

def add_metadata(df):
    """Add standard metadata columns to the dataframe."""
    return (
        df.withColumn("period", F.lit(period))
          .withColumn("run_id", F.lit(run_id))
          .withColumn("dfm_id", F.lit(dfm_id))
          .withColumn("dfm_name", F.lit(dfm_name))
          .withColumn("ingested_at", F.current_timestamp())
    )

if df_positions:
    df_positions = add_metadata(df_positions)
    print(f"[INFO] Metadata added to positions: {df_positions.count()} rows ready")

if df_cash:
    df_cash = add_metadata(df_cash)
    print(f"[INFO] Metadata added to cash: {df_cash.count()} rows ready")

In [ ]:
# ---------- Write to Staging Tables ----------
print("[STEP 6] Writing to staging tables...")

rows_written = 0

if df_positions:
    table_name = "stage1_brown_shipley_positions_raw"
    print(f"[INFO] Writing positions to {table_name}...")
    df_positions.write.mode("append").saveAsTable(table_name)
    pos_count = df_positions.count()
    rows_written += pos_count
    print(f"[SUCCESS] Written {pos_count} position rows")

if df_cash:
    table_name = "stage1_brown_shipley_cash_raw"
    print(f"[INFO] Writing cash to {table_name}...")
    df_cash.write.mode("append").saveAsTable(table_name)
    cash_count = df_cash.count()
    rows_written += cash_count
    print(f"[SUCCESS] Written {cash_count} cash rows")

print(f"[INFO] Total rows written: {rows_written}")

In [ ]:
# ---------- Write Audit Log ----------
print("[STEP 7] Writing audit log...")

try:
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, 
         parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{run_id}', '{period}', '{dfm_id}', {len(csv_files)}, {rows_written}, 
         0, 0, 'OK', current_timestamp(), current_timestamp())
    """)
    print("[SUCCESS] Audit log written")
except Exception as e:
    print(f"[WARNING] Could not write audit log: {str(e)}")

In [ ]:
# ---------- Summary ----------
print("\n" + "=" * 80)
print("[INGESTION COMPLETE]")
print("=" * 80)
print(f"Period: {period}")
print(f"Run ID: {run_id}")
print(f"DFM: {dfm_name}")
print(f"Files processed: {len(csv_files)}")
print(f"Total rows ingested: {rows_written}")
if df_positions:
    print(f"  - Positions: {df_positions.count()} rows")
if df_cash:
    print(f"  - Cash: {df_cash.count()} rows")
print(f"Status: OK")
print("=" * 80)

mssparkutils.notebook.exit("OK")